# research-note_1
## 1 Kolmogorov–Arnold Representation Theorem

$$
f(x_1,\dots,x_n)
=
\sum_{q=0}^{2n}
\Phi_q
\left(
\sum_{p=1}^{n}
\phi_{q,p}(x_p)
\right)
$$

###     worked Example
    xy = exp ( log x + log y ),  x,y > 0 
            


In [17]:
import math 

def xy_via_ka(x , y):
    """
    Reproduce x*y using only univariate functions + addition,
    matching the KA theorem's shape for one term.
    """
    phi1 = math.log(x)
    phi2 = math.log(y)
    inner_sum = phi1 + phi2
    Phi = math.exp(inner_sum)
    return Phi

print(xy_via_ka(3 , 4))  # Should print 12.0

12.0


## 2. MLP - Hidden Layers

$$
h=\sigma(W_1x+b_1),
\qquad
\hat{y}=W_2h+b_2
$$

In [18]:
import torch
import torch.nn as nn

# Minimal 2-layer MLP: input -> hidden(with activation) -> output

mlp = nn.Sequential(
    nn.Linear( in_features = 4 , out_features = 8 ), # W1 x + b1
    nn.ReLU(),  # Activation function, sigma(...)
    nn.Linear( in_features = 8 , out_features = 1 )  # W2 h + b2
)

x = torch.randn(1 , 4) # one sample, 4 input features
y_hat = mlp(x)
print(y_hat)

tensor([[-0.2208]], grad_fn=<AddmmBackward0>)


![4-8-1 兩層 MLP](fig/mlp_4_8_1.svg)

## 3. Activation Functions

$$
\begin{aligned}
\mathrm{ReLU}(z) &= \max(0, z) &&\in [0, \infty)\\[4pt]
\sigma(z) &= \frac{1}{1 + e^{-z}} &&\in (0, 1)\\[4pt]
\tanh(z) &= \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}} &&\in (-1, 1)
\end{aligned}
$$

In [ ]:
import torch
import torch.nn.functional as F

x = torch.linspace(-5, 5, steps=5)

print("ReLU: ", F.relu(x))
print("Sigmoid: ", torch.sigmoid(x))
print("Tanh: ", torch.tanh(x))

ReLU:  tensor([0.0000, 0.0000, 0.0000, 2.5000, 5.0000])
Sigmoid:  tensor([0.0067, 0.0759, 0.5000, 0.9241, 0.9933])
Tanh:  tensor([-0.9999, -0.9866,  0.0000,  0.9866,  0.9999])


## 4. Minibatch Stochastic Gradient Descent
$$
w \leftarrow w - \frac{\eta}{|\mathcal{B}|} \sum_{i \in \mathcal{B}} \partial_w \ell^{(i)}(w, b),
\qquad
b \leftarrow b - \frac{\eta}{|\mathcal{B}|} \sum_{i \in \mathcal{B}} \partial_b \ell^{(i)}(w, b)
$$

```python
import torch

def minibatch_sgd_step(params, lr, batch_size):
    """One SGD update step over a list of parameter tensors."""
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad / batch_size
            p.grad.zero_()

w = torch.zeros((d, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)
params = [w, b]

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        y_hat = X @ w + b
        l = ((y_hat - y) ** 2 / 2)
        l.sum().backward()
        minibatch_sgd_step(params, lr, batch_size)
```

In [20]:
import torch
import random

def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices)
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(indices[i:i + batch_size])
        yield features[batch_indices], labels[batch_indices]

d, lr, batch_size, num_epochs = 2, 0.03, 10, 3

## 5. Splines - Intro
$$\text{spline}(x) = \sum_i c_i B_i(x)$$
$$\frac{\partial\,\text{spline}(x)}{\partial c_i} = B_i(x)$$

In [21]:
import torch

def spline_eval(x, basis_fns, coeffs):
    """spline(x) = sum_i c_i * B_i(x)"""
    return sum(c * b(x) for c, b in zip(coeffs, basis_fns))